# 🏗️ AI-Based Tower Crane Placement Optimization Using a Neural Network Surrogate

Train an MLP to estimate candidate-location cost, search it with a genetic algorithm, and verify the final location with exact geometry.

> Educational planning demonstration—not a lift plan or safety approval.

👉 **Open the interactive companion:** [https://tower-crane-placement.streamlit.app](https://tower-crane-placement.streamlit.app/?stage=start)

## Complete workflow

Site geometry → exact candidate costs → MLP surrogate → genetic search → exact verification → before/after report.

## Interactive learning journey

- [Where Should the Tower Crane Stand?](https://tower-crane-placement.streamlit.app/?stage=problem) — Placement Optimization Objective
- [Lift Points and Daily Frequency](https://tower-crane-placement.streamlit.app/?stage=demand) — Weighted Demand
- [Reach and Unsafe Areas](https://tower-crane-placement.streamlit.app/?stage=constraints) — Feasibility Constraints
- [The Exact Placement Score](https://tower-crane-placement.streamlit.app/?stage=cost) — Engineering Objective Function
- [Thousands of Candidate Locations](https://tower-crane-placement.streamlit.app/?stage=dataset) — Surrogate Training Data
- [Learning How Good a Position Is](https://tower-crane-placement.streamlit.app/?stage=surrogate) — MLP Cost Surrogate
- [Checking the Learned Cost Surface](https://tower-crane-placement.streamlit.app/?stage=audit) — Surrogate Error Audit
- [Evolving Better Crane Positions](https://tower-crane-placement.streamlit.app/?stage=search) — Genetic Algorithm
- [The Verified Crane Recommendation](https://tower-crane-placement.streamlit.app/?stage=verify) — Exact Post-Optimization Check

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
SITE=(80,60);LIFTS=np.array([[20,15],[55,20],[40,45]],float);FREQ=np.array([30,15,40]);REACH=45;OBST=(25,32,25,38);SAFE=(5,22,2,10)

---
# 1. Where Should the Tower Crane Stand?
### Phase 1 of 6 · The Site Logistics Problem

## Part 1 · On the construction site
A tower crane must serve material-delivery points scattered across a bounded construction site.

## Part 2 · The engineering challenge
A poor location increases slewing and trolley travel, may leave lifts beyond reach, and can conflict with obstacles or prohibited zones.

## Part 3 · Where the AI comes in
Search candidate coordinates for the lowest feasible weighted lifting effort.

**Construction Planning:** Where Should the Tower Crane Stand? → **AI:** Placement Optimization Objective → `recommended X,Y coordinates`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=problem](https://tower-crane-placement.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

The output is an X,Y coordinate. The surrogate predicts a score; it does not directly output or certify the final position. Exact verification remains mandatory.

## Part 5 · What you just built

**In the notebook:** Define the 80×60 m site and placement output.

**Takeaway:** The recommendation is a coordinate supported by an explicit objective and constraints.

[Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Lift Points and Daily Frequency](https://tower-crane-placement.streamlit.app/?stage=demand) ▶

---
# 2. Lift Points and Daily Frequency
### Phase 1 of 6 · The Site Logistics Problem

## Part 1 · On the construction site
A concrete core requiring forty lifts per day should influence placement more than a delivery point used three times.

## Part 2 · The engineering challenge
Minimizing unweighted average distance can optimize rare lifts while penalizing repetitive site logistics.

## Part 3 · Where the AI comes in
Multiply each crane-to-lift distance by its expected lift frequency.

**Construction Planning:** Lift Points and Daily Frequency → **AI:** Weighted Demand → `Σ frequency × distance`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=demand](https://tower-crane-placement.streamlit.app/?stage=demand)

## Part 4 · The technical explanation

In [ ]:
def weighted_distance(p):
 d=np.linalg.norm(LIFTS-p,axis=1);return float((FREQ*d).sum()),d
for p in [np.array([15,20]),np.array([40,30]),np.array([65,25])]:print(p,weighted_distance(p))

## Part 5 · What you just built

**In the notebook:** Create lift coordinates, frequencies, and weighted-distance calculations.

**Takeaway:** Frequency converts geometric distance into operational lifting effort.

◀ [Previous: Where Should the Tower Crane Stand?](https://tower-crane-placement.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Reach and Unsafe Areas](https://tower-crane-placement.streamlit.app/?stage=constraints) ▶

---
# 3. Reach and Unsafe Areas
### Phase 2 of 6 · Engineering Constraints

## Part 1 · On the construction site
The crane base cannot occupy a building, temporary structure, or exclusion zone, and required lift points must lie within boom reach.

## Part 2 · The engineering challenge
A mathematically short location is useless if it is unsafe or cannot reach the workface.

## Part 3 · Where the AI comes in
Separate feasibility checks from the distance objective and penalize violations strongly during search.

**Construction Planning:** Reach and Unsafe Areas → **AI:** Feasibility Constraints → `boom radius + obstacle and safety-zone checks`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=constraints](https://tower-crane-placement.streamlit.app/?stage=constraints)

## Part 4 · The technical explanation

In [ ]:
def inside(p,rect):return rect[0]<=p[0]<=rect[1] and rect[2]<=p[1]<=rect[3]
def feasibility(p):
 effort,d=weighted_distance(p);return dict(boundary=0<=p[0]<=80 and 0<=p[1]<=60,obstacle=inside(p,OBST),safety_zone=inside(p,SAFE),reachable=d<=REACH,distances=d)
for p in [np.array([15,20]),np.array([40,30]),np.array([65,25])]:print(p,feasibility(p))

## Part 5 · What you just built

**In the notebook:** Represent rectangles, boom reach, site boundary, and reachability.

**Takeaway:** Constraints define what can be built; optimization chooses among feasible positions.

◀ [Previous: Lift Points and Daily Frequency](https://tower-crane-placement.streamlit.app/?stage=demand) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Exact Placement Score](https://tower-crane-placement.streamlit.app/?stage=cost) ▶

---
# 4. The Exact Placement Score
### Phase 2 of 6 · Engineering Constraints

## Part 1 · On the construction site
Planners need a transparent score that distinguishes efficient feasible points from unsafe or unreachable points.

## Part 2 · The engineering challenge
Penalty values must dominate normal travel cost without hiding which constraint failed.

## Part 3 · Where the AI comes in
Calculate exact distance cost and attach named penalties for each violation.

**Construction Planning:** The Exact Placement Score → **AI:** Engineering Objective Function → `weighted distance + unreachable and safety penalties`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=cost](https://tower-crane-placement.streamlit.app/?stage=cost)

## Part 4 · The technical explanation

In [ ]:
def exact_cost(p,details=False):
 effort,d=weighted_distance(p);f=feasibility(p);penalty=15000*(f["obstacle"]+f["safety_zone"]+int(not f["boundary"]))+10000*(~f["reachable"]).sum();total=effort+penalty
 return (total,effort,penalty,f) if details else total
for p in [np.array([15,20]),np.array([40,30]),np.array([65,25])]:print(p,exact_cost(p,True))

## Part 5 · What you just built

**In the notebook:** Implement the exact evaluator and test several candidate positions.

**Takeaway:** A cost function is an engineering specification written as arithmetic.

◀ [Previous: Reach and Unsafe Areas](https://tower-crane-placement.streamlit.app/?stage=constraints) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Thousands of Candidate Locations](https://tower-crane-placement.streamlit.app/?stage=dataset) ▶

---
# 5. Thousands of Candidate Locations
### Phase 3 of 6 · Learning the Cost Surface

## Part 1 · On the construction site
Evaluating a dense candidate grid or many evolving layouts can require repeated geometric checks.

## Part 2 · The engineering challenge
A neural surrogate is only useful if its training candidates cover safe, unsafe, reachable, and boundary regions.

## Part 3 · Where the AI comes in
Sample thousands of positions, compute exact costs, and train on the resulting candidate table.

**Construction Planning:** Thousands of Candidate Locations → **AI:** Surrogate Training Data → `candidate X,Y + fixed site encoding -> exact cost`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=dataset](https://tower-crane-placement.streamlit.app/?stage=dataset)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(SEED);candidates=np.column_stack([rng.uniform(0,80,10000),rng.uniform(0,60,10000)]);targets=np.array([exact_cost(p) for p in candidates],dtype="float32")
data=pd.DataFrame(dict(x=candidates[:,0],y=candidates[:,1],cost=targets));print(data.describe());plt.scatter(candidates[:,0],candidates[:,1],c=np.log1p(targets),s=5,cmap="viridis");plt.colorbar(label="log exact cost");plt.xlabel("X");plt.ylabel("Y");plt.show()

## Part 5 · What you just built

**In the notebook:** Generate candidate coordinates and exact labels across the site.

**Takeaway:** The surrogate can only learn portions of the cost surface represented in its examples.

◀ [Previous: The Exact Placement Score](https://tower-crane-placement.streamlit.app/?stage=cost) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Learning How Good a Position Is](https://tower-crane-placement.streamlit.app/?stage=surrogate) ▶

---
# 6. Learning How Good a Position Is
### Phase 3 of 6 · Learning the Cost Surface

## Part 1 · On the construction site
The same geometric evaluator may be called repeatedly by a search algorithm.

## Part 2 · The engineering challenge
The cost surface contains sharp penalty boundaries that a smooth network can approximate poorly.

## Part 3 · Where the AI comes in
Train an MLP regressor to estimate cost quickly while retaining exact verification for final decisions.

**Construction Planning:** Learning How Good a Position Is → **AI:** MLP Cost Surrogate → `candidate/site features -> predicted cost`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=surrogate](https://tower-crane-placement.streamlit.app/?stage=surrogate)

## Part 4 · The technical explanation

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(candidates,targets,test_size=.20,random_state=SEED);scaler=StandardScaler().fit(Xtr);Xtr_s,Xte_s=scaler.transform(Xtr),scaler.transform(Xte);ys=StandardScaler().fit(ytr.reshape(-1,1));ytr_s=ys.transform(ytr.reshape(-1,1))
model=Sequential([Input((2,)),Dense(64,activation="relu"),Dense(64,activation="relu"),Dense(32,activation="relu"),Dense(1)]);model.compile(optimizer="adam",loss="mae");early=EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True);model.fit(Xtr_s,ytr_s,validation_split=.15,epochs=70,batch_size=96,callbacks=[early],verbose=0)
def predict_cost(points):return ys.inverse_transform(model.predict(scaler.transform(points),verbose=0)).ravel()
print(model.summary())

## Part 5 · What you just built

**In the notebook:** Build and train the surrogate, then plot predicted cost contours.

**Takeaway:** A surrogate accelerates search; it does not replace the exact engineering evaluator.

◀ [Previous: Thousands of Candidate Locations](https://tower-crane-placement.streamlit.app/?stage=dataset) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Checking the Learned Cost Surface](https://tower-crane-placement.streamlit.app/?stage=audit) ▶

---
# 7. Checking the Learned Cost Surface
### Phase 3 of 6 · Learning the Cost Surface

## Part 1 · On the construction site
Placement depends more on selecting the right low-cost region than matching every huge penalty exactly.

## Part 2 · The engineering challenge
A low global error can hide poor ranking among feasible candidates or false predictions near exclusion boundaries.

## Part 3 · Where the AI comes in
Measure overall error, feasible-only error, top-candidate ranking agreement, and boundary cases.

**Construction Planning:** Checking the Learned Cost Surface → **AI:** Surrogate Error Audit → `MAE, safe-region error, ranking agreement`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=audit](https://tower-crane-placement.streamlit.app/?stage=audit)

## Part 4 · The technical explanation

In [ ]:
pred=predict_cost(Xte);print("Overall MAE:",mean_absolute_error(yte,pred));feasible=yte<10000;print("Feasible-region MAE:",mean_absolute_error(yte[feasible],pred[feasible]));print("RMSE:",mean_squared_error(yte,pred)**.5)
plt.scatter(yte,pred,s=7,alpha=.25);lim=max(yte.max(),pred.max());plt.plot([0,lim],[0,lim],"r--");plt.xlabel("Exact cost");plt.ylabel("Surrogate cost");plt.show()
exact_top=set(np.argsort(yte)[:100]);pred_top=set(np.argsort(pred)[:100]);print("Top-100 ranking overlap:",len(exact_top&pred_top),"%")

## Part 5 · What you just built

**In the notebook:** Compare predicted and exact costs on unseen positions.

**Takeaway:** Audit the decisions the surrogate will influence, not only its average numeric error.

◀ [Previous: Learning How Good a Position Is](https://tower-crane-placement.streamlit.app/?stage=surrogate) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Evolving Better Crane Positions](https://tower-crane-placement.streamlit.app/?stage=search) ▶

---
# 8. Evolving Better Crane Positions
### Phase 4 of 6 · Searching for Placement

## Part 1 · On the construction site
The planner wants a good position without checking every centimetre of the site.

## Part 2 · The engineering challenge
Search can converge to a surrogate error or an unsafe boundary if constraints are not respected.

## Part 3 · Where the AI comes in
Evolve populations on predicted cost, clip to site limits, and preserve diverse candidates.

**Construction Planning:** Evolving Better Crane Positions → **AI:** Genetic Algorithm → `select, crossover, mutate candidate coordinates`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=search](https://tower-crane-placement.streamlit.app/?stage=search)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(11);pop=np.column_stack([rng.uniform(0,80,120),rng.uniform(0,60,120)]);history=[]
for generation in range(55):
 scores=predict_cost(pop);order=np.argsort(scores);elite=pop[order[:24]];history.append(scores[order[0]]);parents=elite[rng.integers(0,len(elite),(96,2))];children=(parents[:,0]+parents[:,1])/2+rng.normal(0,[3,2],(96,2));children[:,0]=np.clip(children[:,0],0,80);children[:,1]=np.clip(children[:,1],0,60);pop=np.vstack([elite,children])
plt.plot(history);plt.xlabel("Generation");plt.ylabel("Best predicted cost");plt.grid(alpha=.2);plt.show();print("Best surrogate candidates:",pop[np.argsort(predict_cost(pop))[:5]])

## Part 5 · What you just built

**In the notebook:** Run a compact GA and plot best cost by generation.

**Takeaway:** The optimizer proposes; the exact evaluator verifies.

◀ [Previous: Checking the Learned Cost Surface](https://tower-crane-placement.streamlit.app/?stage=audit) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Verified Crane Recommendation](https://tower-crane-placement.streamlit.app/?stage=verify) ▶

---
# 9. The Verified Crane Recommendation
### Phase 5 of 6 · Verifying the Recommendation

## Part 1 · On the construction site
Before a location is reported, planners need its reach coverage, safety status, and lifting-effort metrics.

## Part 2 · The engineering challenge
A surrogate optimum is not acceptable until the original objective and every constraint are recalculated.

## Part 3 · Where the AI comes in
Evaluate the best GA candidates exactly, choose the best feasible one, and compare it with a baseline position.

**Construction Planning:** The Verified Crane Recommendation → **AI:** Exact Post-Optimization Check → `coordinates, reach, violations, exact cost`

> 🎬 **See this illustrated and interactive:** [https://tower-crane-placement.streamlit.app/?stage=verify](https://tower-crane-placement.streamlit.app/?stage=verify)

## Part 4 · The technical explanation

In [ ]:
# Recheck the best surrogate candidates with the original exact evaluator.
shortlist=pop[np.argsort(predict_cost(pop))[:30]];verified=sorted([(exact_cost(p),p,exact_cost(p,True)) for p in shortlist],key=lambda x:x[0]);best_cost,best,details=verified[0];baseline=np.array([15.,20.]);base_details=exact_cost(baseline,True)
print(f"OPTIMAL VERIFIED CRANE POSITION: X={best[0]:.1f} m, Y={best[1]:.1f} m");print("Weighted lifting effort:",details[1]);print("Reachable lift points:",100*details[3]["reachable"].mean(),"%");print("Safety-zone violations:",int(details[3]["obstacle"])+int(details[3]["safety_zone"]));print("Exact cost:",best_cost)
comparison=pd.DataFrame({"Metric":["Weighted effort","Reachable points","Safety violations","Exact total cost"],"Baseline":[base_details[1],f"{100*base_details[3]['reachable'].mean():.0f}%",int(base_details[3]["obstacle"])+int(base_details[3]["safety_zone"]),base_details[0]],"Optimized":[details[1],f"{100*details[3]['reachable'].mean():.0f}%",int(details[3]["obstacle"])+int(details[3]["safety_zone"]),best_cost]});display(comparison)
print("Limitations: 2D distance proxy only; no load chart, hook height, cycle time, slew collision, foundation, ties, power lines, sequencing, multiple cranes, wind, or regulatory planning.")

## Part 5 · What you just built

**In the notebook:** Display before/after maps, reach circle, metrics, and limitations.

**Takeaway:** Only an exactly verified feasible candidate becomes the educational recommendation.

◀ [Previous: Evolving Better Crane Positions](https://tower-crane-placement.streamlit.app/?stage=search) &nbsp;|&nbsp; [Project overview](https://tower-crane-placement.streamlit.app/?stage=start)

---
# Final engineering conclusion

The MLP surrogate accelerates exploration, the genetic algorithm proposes low-cost coordinates, and the original exact evaluator decides which feasible candidate is reported. This separation prevents a learned approximation from silently overriding site constraints.